In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("--- Building the Ultimate Feature Matrix ---")

# 1. Load Data
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')

# Save target and IDs
target = train['Drafted']
train_ids = train['Id']
test_ids = test['Id']

# Concatenate for uniform feature engineering
# We drop 'Drafted' from train so it perfectly matches test shape
df = pd.concat([train.drop(columns=['Drafted']), test], axis=0).reset_index(drop=True)

# 2. Missing Indicators
physical_metrics = ['Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump',
                    'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle', 'Age']

for col in physical_metrics:
    if df[col].isnull().sum() > 0:
        df[f'{col}_Is_Missing'] = df[col].isnull().astype(int)

# 3. Clean Imputation (Positional Median)
for col in physical_metrics:
    df[col] = df[col].fillna(df.groupby('Position')[col].transform('median'))
    # Fallback in case an entire position is missing a metric
    df[col] = df[col].fillna(df[col].median())

# 4. Core Engineered Features
df['BMI'] = df['Weight'] / (df['Height'] ** 2)
df['Speed_Weight_Ratio'] = df['Weight'] / df['Sprint_40yd']
df['Explosiveness_Index'] = df['Vertical_Jump'] * df['Broad_Jump']
df['Power_Score'] = df['Bench_Press_Reps'] * df['Weight']
df['School_Combine_Count'] = df.groupby('School')['Id'].transform('count')

# 5. Elite Features (Football Outsiders & Positional Relativism)
# Speed Score heavily rewards heavy players who run incredibly fast times
df['Speed_Score'] = (df['Weight'] * 200) / (df['Sprint_40yd'] ** 4)

# Positional Difference: How much better/worse is this player than the average at their position?
for col in physical_metrics:
    # Calculate the mean for the position
    df[f'{col}_Pos_Mean'] = df.groupby('Position')[col].transform('mean')
    # Subtract the position mean from the player's actual score
    df[f'{col}_Pos_Diff'] = df[col] - df[f'{col}_Pos_Mean']
    # Drop the mean column to avoid redundancy, keep the difference
    df.drop(columns=[f'{col}_Pos_Mean'], inplace=True)

# 6. Categorical Casting
categorical_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
for col in categorical_cols:
    df[col] = df[col].astype('category')

# 7. Split back into Train and Test
X_train = df[df['Id'].isin(train_ids)].drop(columns=['Id']).reset_index(drop=True)
X_test = df[df['Id'].isin(test_ids)].drop(columns=['Id']).reset_index(drop=True)
y_train = target.reset_index(drop=True)

print(f"Ultimate X_train shape: {X_train.shape}")
print(f"Ultimate X_test shape: {X_test.shape}")
print(f"Total Features Engineered: {X_train.shape[1]}")

--- Building the Ultimate Feature Matrix ---
Ultimate X_train shape: (2781, 36)
Ultimate X_test shape: (696, 36)
Total Features Engineered: 36


In [3]:
import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

print("--- Phase 2: Deep Optuna Search on Ultimate Matrix ---")

# Define categorical columns explicitly for CatBoost
cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']

# Cross-validation setup exactly matching our target distribution
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    # Deep search space optimized for our 36 features
    params = {
        'iterations': trial.suggest_int('iterations', 600, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'random_seed': 42,
        'verbose': 0
    }

    oof_preds = np.zeros(len(X_train))

    for train_idx, valid_idx in skf.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

        model = CatBoostClassifier(**params, cat_features=cat_cols)
        model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            early_stopping_rounds=50
        )
        oof_preds[valid_idx] = model.predict_proba(X_va)[:, 1]

    return roc_auc_score(y_train, oof_preds)

# 15 trials for our live pipeline (you can boost this to 50+ when running entirely offline later)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

print("\n--- Deep Search Complete ---")
print("Best Parameters:", study.best_params)
print(f"Best OOF ROC-AUC: {study.best_value:.5f}")

[I 2026-05-22 04:50:36,112] A new study created in memory with name: no-name-3753dcd8-1a57-4e6b-8b16-4b58a4aa0911


--- Phase 2: Deep Optuna Search on Ultimate Matrix ---


[I 2026-05-22 04:51:21,312] Trial 0 finished with value: 0.836712727140746 and parameters: {'iterations': 1183, 'learning_rate': 0.06727727086369523, 'depth': 8, 'l2_leaf_reg': 3.549725266707691, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.836712727140746.
[I 2026-05-22 04:52:57,370] Trial 1 finished with value: 0.8412926876020085 and parameters: {'iterations': 1183, 'learning_rate': 0.010492928203031996, 'depth': 4, 'l2_leaf_reg': 4.5594175959311425, 'min_data_in_leaf': 7}. Best is trial 1 with value: 0.8412926876020085.
[I 2026-05-22 04:53:42,653] Trial 2 finished with value: 0.8391994936863918 and parameters: {'iterations': 756, 'learning_rate': 0.031597804915093, 'depth': 6, 'l2_leaf_reg': 2.3280025166226874, 'min_data_in_leaf': 32}. Best is trial 1 with value: 0.8412926876020085.
[I 2026-05-22 04:54:06,170] Trial 3 finished with value: 0.8398556371056194 and parameters: {'iterations': 1059, 'learning_rate': 0.0661983704696613, 'depth': 4, 'l2_leaf_reg': 2.116930281682473


--- Deep Search Complete ---
Best Parameters: {'iterations': 1183, 'learning_rate': 0.010492928203031996, 'depth': 4, 'l2_leaf_reg': 4.5594175959311425, 'min_data_in_leaf': 7}
Best OOF ROC-AUC: 0.84129


In [4]:
import pandas as pd
from catboost import CatBoostClassifier

print("--- Phase 3: Training Final Ultimate Model ---")

# The exact best parameters from your Deep Optuna search
ultimate_params = {
    'iterations': 1183,
    'learning_rate': 0.010492928203031996,
    'depth': 4,
    'l2_leaf_reg': 4.5594175959311425,
    'min_data_in_leaf': 7,
    'random_seed': 42,
    'verbose': 100 # Print progress every 100 trees
}

# Define categorical columns explicitly for CatBoost
cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']

# Initialize the model
ultimate_model = CatBoostClassifier(**ultimate_params, cat_features=cat_cols)

# Train on 100% of the training data (X_train and y_train from Phase 1)
ultimate_model.fit(X_train, y_train)

print("\n--- Generating Final Predictions ---")
# Predict probabilities for class 1.0 (Drafted)
test_preds = ultimate_model.predict_proba(X_test)[:, 1]

# Format exactly as Kaggle expects
submission = pd.DataFrame({
    'Id': test_ids,
    'Drafted': test_preds
})

submission.to_csv('Data/Submission NFL 1 Optimized.csv', index=False)
print("SUCCESS: 'Submission NFL 1 Optimized.csv' has been generated and is ready to submit!")

--- Phase 3: Training Final Ultimate Model ---
0:	learn: 0.6870787	total: 17.5ms	remaining: 20.7s
100:	learn: 0.4629018	total: 2.99s	remaining: 32s
200:	learn: 0.4312960	total: 5.15s	remaining: 25.2s
300:	learn: 0.4162650	total: 7.33s	remaining: 21.5s
400:	learn: 0.4069500	total: 10.8s	remaining: 21s
500:	learn: 0.3992502	total: 13.8s	remaining: 18.8s
600:	learn: 0.3930499	total: 17.9s	remaining: 17.3s
700:	learn: 0.3872184	total: 20.1s	remaining: 13.8s
800:	learn: 0.3819289	total: 22.2s	remaining: 10.6s
900:	learn: 0.3767346	total: 24.5s	remaining: 7.67s
1000:	learn: 0.3722938	total: 26.6s	remaining: 4.84s
1100:	learn: 0.3674513	total: 28.7s	remaining: 2.13s
1182:	learn: 0.3635553	total: 30.7s	remaining: 0us

--- Generating Final Predictions ---
SUCCESS: 'submission_ultimate.csv' has been generated and is ready to submit!
